Add attention

In [1]:
from tensorflow.keras.layers import Attention

2026-04-06 17:02:57.869411: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775494978.104011      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775494978.169506      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775494978.696253      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775494978.696319      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775494978.696322      24 computation_placer.cc:177] computation placer alr

In [2]:
import pandas as pd
import os

DATA_PATH = "/kaggle/input/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/cnn_dailymail"

train_file = os.path.join(DATA_PATH, "train.csv")

df = pd.read_csv(train_file)

df = df[['article', 'highlights']]
df = df.dropna()

# Keep SAME size as baseline
df = df.head(30000)

# Add tokens
df['summary'] = df['highlights'].apply(lambda x: '<start> ' + x + ' <end>')

df.head()

,article,highlights,summary
0,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ...","<start> Bishop John Folda, of North Dakota, is..."
1,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...,<start> Criminal complaint: Cop used his role ...
2,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t...","<start> Craig Eccleston-Todd, 27, had drunk at..."
3,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...,<start> Nina dos Santos says Europe must be re...
4,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...,<start> Fleetwood top of League One after 2-0 ...


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# IMPORTANT: keep < >
tgt_tokenizer = Tokenizer(num_words=15000, filters='')
src_tokenizer = Tokenizer(num_words=30000)

src_tokenizer.fit_on_texts(df['article'])
tgt_tokenizer.fit_on_texts(df['summary'])

encoder_seq = src_tokenizer.texts_to_sequences(df['article'])
decoder_seq = tgt_tokenizer.texts_to_sequences(df['summary'])

# SAME lengths
MAX_LEN_SRC = 300
MAX_LEN_TGT = 40

encoder_seq = pad_sequences(encoder_seq, maxlen=MAX_LEN_SRC, padding='post')
decoder_seq = pad_sequences(decoder_seq, maxlen=MAX_LEN_TGT, padding='post')

In [4]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention, Concatenate

latent_dim = 256

# ======================
# 🔹 ENCODER
# ======================
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(30000, latent_dim, mask_zero=False)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# ======================
# 🔹 DECODER
# ======================
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(15000, latent_dim, mask_zero=False)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# ======================
# 🔥 ATTENTION
# ======================
attention_layer = Attention()
attention_output = attention_layer([decoder_outputs, encoder_outputs])

# Combine context + decoder
concat = Concatenate(axis=-1)([decoder_outputs, attention_output])

# Output layer
decoder_dense = Dense(15000, activation='softmax')
decoder_outputs = decoder_dense(concat)

# ======================
# MODEL
# ======================
model_attn = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model_attn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

model_attn.summary()

I0000 00:00:1775495073.523298      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775495073.529275      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  7,680,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  3,840,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │    525,312 │ embedding[0][0]   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    525,312 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 256) │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 512) │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  7,695,000 │ concatenate[0][0] │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,265,624 (77.31 MB)

 Trainable params: 20,265,624 (77.31 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import numpy as np

decoder_input = decoder_seq[:, :-1]
decoder_output = decoder_seq[:, 1:]

decoder_output = np.expand_dims(decoder_output, -1)

In [6]:
model_attn.fit(
    [encoder_seq, decoder_input],
    decoder_output,
    batch_size=64,
    epochs=100,  # SAME as baseline
    validation_split=0.2
)

Epoch 1/100


I0000 00:00:1775495078.808670      73 cuda_dnn.cc:529] Loaded cuDNN version 91002


375/375 ━━━━━━━━━━━━━━━━━━━━ 48s 118ms/step - loss: 6.9191 - val_loss: 6.0675
Epoch 2/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 47s 126ms/step - loss: 5.8931 - val_loss: 5.6324
Epoch 3/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 51s 136ms/step - loss: 5.4341 - val_loss: 5.4060
Epoch 4/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 51s 136ms/step - loss: 5.1385 - val_loss: 5.2818
Epoch 5/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 51s 137ms/step - loss: 4.8968 - val_loss: 5.2059
Epoch 6/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 51s 137ms/step - loss: 4.6816 - val_loss: 5.1619
Epoch 7/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 52s 138ms/step - loss: 4.4983 - val_loss: 5.1478
Epoch 8/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 52s 138ms/step - loss: 4.3246 - val_loss: 5.1591
Epoch 9/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 51s 137ms/step - loss: 4.1382 - val_loss: 5.1707
Epoch 10/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 52s 138ms/step - loss: 3.9687 - val_loss: 5.2125
Epoch 11/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 52s 139ms/step - loss: 3.7918 - val_loss: 5.2593
Epoch 12/100
375/375

In [7]:
model_attn.save_weights("summarizer_attn.weights.h5")

import pickle
pickle.dump(src_tokenizer, open("src_tok.pkl", "wb"))
pickle.dump(tgt_tokenizer, open("tgt_tok.pkl", "wb"))

In [8]:
# Encoder
encoder_model_attn = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# Decoder inputs
from tensorflow.keras.layers import Input

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
encoder_outputs_input = Input(shape=(MAX_LEN_SRC, latent_dim))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

# Attention again
attention_out2 = attention_layer([decoder_outputs2, encoder_outputs_input])
concat2 = Concatenate(axis=-1)([decoder_outputs2, attention_out2])

decoder_outputs2 = decoder_dense(concat2)

decoder_model_attn = Model(
    [decoder_inputs, encoder_outputs_input] + decoder_states_inputs,
    [decoder_outputs2, state_h2, state_c2]
)

In [9]:
reverse_tgt_index = {i: w for w, i in tgt_tokenizer.word_index.items()}

def summarize_attn(text):
    seq = src_tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=300, padding='post')

    enc_out, h, c = encoder_model_attn.predict(seq)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = tgt_tokenizer.word_index['<start>']

    summary = []

    for _ in range(40):
        output, h, c = decoder_model_attn.predict(
            [target_seq, enc_out, h, c]
        )

        idx = np.argmax(output[0,-1,:])
        word = reverse_tgt_index.get(idx, '')

        if word == '<end>' or word == '':
            break

        summary.append(word)

        target_seq[0,0] = idx

    return " ".join(summary)

In [10]:
sample = df['article'].iloc[0]

print("INPUT:\n", sample)
print("\nSUMMARY:\n", summarize_attn(sample))

INPUT:
 By . Associated Press . PUBLISHED: . 14:11 EST, 25 October 2013 . | . UPDATED: . 15:36 EST, 25 October 2013 . The bishop of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo, Grand Forks and Jamestown to the hepatitis A virus in late September and early October. The state Health Department has issued an advisory of exposure for anyone who attended five churches and took communion. Bishop John Folda (pictured) of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo, Grand Forks and Jamestown to the hepatitis A . State Immunization Program Manager Molly Howell says the risk is low, but officials feel it's important to alert people to the possible exposure. The diocese announced on Monday that Bishop John Folda is taking time off after being diagnosed with hepatitis A. The diocese says he contracted the infection through contaminated food while attending a conference for newly or